# 📚 Data I/O — Reading & Writing CSV and Excel Files
### 데이터 불러오기 / 저장 — CSV와 Excel 파일

> **Section 2 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA
> 전체 11개 섹션 중 **2번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to read CSV files cleanly on the first try — encoding, custom missing-value markers, dates, and thousands separators / CSV 파일을 처음부터 깔끔하게 읽는 방법 — 인코딩, 커스텀 결측치 표시, 날짜, 천단위 구분자
- [x] How to read one Excel sheet, all sheets at once, and write multiple sheets back out / Excel 시트 하나, 모든 시트를 한 번에 읽고, 여러 시트로 다시 저장하는 방법
- [x] The standard "clean-on-load" pattern that combines every option into a single `read_csv()` call / 모든 옵션을 하나의 `read_csv()` 호출로 합치는 표준 "불러오면서 정리하기" 패턴

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
Data I/O is the very first step of any analysis: turning a file that lives outside Python — a CSV exported from a database, an Excel workbook someone emailed you — into a DataFrame you can work with, and later turning your cleaned DataFrame back into a file someone else can open. `pd.read_csv()` / `df.to_csv()` and `pd.read_excel()` / `df.to_excel()` are the two most common pairs.

**한글**
데이터 I/O는 모든 분석의 첫 단계입니다: 데이터베이스에서 내보낸 CSV, 누군가 메일로 보낸 엑셀 파일처럼 파이썬 바깥에 있는 파일을 DataFrame으로 바꾸고, 나중에는 정리한 DataFrame을 다른 사람이 열어볼 수 있는 파일로 다시 바꾸는 것입니다. `pd.read_csv()` / `df.to_csv()`와 `pd.read_excel()` / `df.to_excel()`이 가장 흔한 두 쌍입니다.

## Why do we use it?
*(When is it useful?)*

**English**
Real files are messy in ways that JSON from an API rarely is: dates arrive as plain text, numbers arrive with thousands-separator commas, missing values are marked with things like `"-"` or `"N/A"` instead of being truly empty cells, and Korean text saved on Windows can arrive in a different encoding than expected. `read_csv()` / `read_excel()` give you parameters — `na_values`, `parse_dates`, `thousands`, `dtype`, `encoding` — to handle every one of these on the same line you load the file, instead of cleaning them up one by one afterward.

**한글**
실제 파일은 API의 JSON과 달리 지저분한 경우가 많습니다: 날짜가 순수 텍스트로 들어오고, 숫자에 천단위 구분 쉼표가 붙어 있고, 결측치가 진짜 빈 셀이 아니라 `"-"`나 `"N/A"` 같은 문자로 표시되어 있고, Windows에서 저장된 한글 텍스트는 예상과 다른 인코딩으로 들어올 수 있습니다. `read_csv()` / `read_excel()`은 `na_values`, `parse_dates`, `thousands`, `dtype`, `encoding` 같은 파라미터를 제공해서, 파일을 불러오는 바로 그 줄에서 이 모든 문제를 처리할 수 있게 해줍니다 — 나중에 하나씩 따로 정리할 필요 없이.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
This is where essentially every real analysis begins — before you can group, filter, or chart anything, the raw export has to become a DataFrame with the right dtypes. Getting `read_csv()`'s options right the first time is the difference between "the amount column summed into a weird string" and a clean five-minute analysis. This step also repeats constantly: a new CSV lands every morning, a new sheet gets added to the tracking workbook every month.

**한글**
실질적으로 모든 실제 분석이 여기서 시작합니다 — 그룹화, 필터링, 차트 작업을 하기 전에 원본 내보내기 파일이 올바른 dtype을 가진 DataFrame이 되어야 합니다. `read_csv()`의 옵션을 처음부터 제대로 지정하는 것이 "amount 열을 합쳤더니 이상한 문자열이 나왔다"와 깔끔한 5분짜리 분석의 차이를 만듭니다. 이 단계는 계속 반복됩니다 — 매일 아침 새 CSV가 들어오고, 매달 관리용 워크북에 새 시트가 추가됩니다.

### Quick Comparison: JS/TS vs pandas / 빠른 비교

| Concept / 개념 | JavaScript / TypeScript | pandas |
|---|---|---|
| Read a CSV / CSV 읽기 | `fetch()` + a parsing library (e.g. Papa Parse) / `fetch()` + 파싱 라이브러리 | `pd.read_csv("file.csv")` — built in / 내장 |
| Handle a custom "missing" marker / 커스텀 결측 표시 처리 | manual `.map()` after parsing / 파싱 후 수동 `.map()` | `na_values=["-", "N/A"]` parameter |
| Parse a date column / 날짜 열 파싱 | `new Date(str)` per row / 행마다 `new Date(str)` | `parse_dates=["date_col"]` parameter |
| Strip thousands separators / 천단위 구분자 제거 | manual `.replace(",", "")` per row / 행마다 수동 `.replace(",","")` | `thousands=","` parameter |
| Read one Excel sheet / 시트 하나 읽기 | a separate library (e.g. SheetJS) / 별도 라이브러리 필요 | `pd.read_excel("f.xlsx", sheet_name="Sheet1")` |
| Read every Excel sheet at once / 모든 시트 한번에 읽기 | loop over sheet names manually / 시트 이름을 직접 순회 | `pd.read_excel("f.xlsx", sheet_name=None)` → dict of DataFrames |

---
# 📝 Syntax

## Basic Syntax

In [2]:
import pandas as pd
from io import StringIO

# The base pattern for a REAL file on disk (this line needs an actual file, so it stays as a comment here)
# 실제 디스크에 있는 파일을 읽는 기본 패턴 (실제 파일이 필요하므로 여기서는 주석으로만 표시)
# df = pd.read_csv("sales.csv")
# df.to_csv("output.csv", index=False)

# This notebook has no file attached, so every runnable example below builds a
# "virtual file" in memory with StringIO -- every cell actually executes and shows real output.
# 이 노트북에는 첨부된 파일이 없으므로, 아래의 모든 실행 예제는 StringIO로 메모리 안에
# "가상 파일"을 만듭니다 -- 모든 셀이 실제로 실행되고 진짜 결과를 보여줍니다.
csv_text = """name,score
Alice,85
Bob,92
Charlie,78"""

df = pd.read_csv(StringIO(csv_text))
print(df)
print(df.dtypes)

      name  score
0    Alice     85
1      Bob     92
2  Charlie     78
name       str
score    int64
dtype: object


## Common Variations

In [3]:
import pandas as pd
from io import StringIO

csv_text = """name,score
Alice,85
Bob,92
Charlie,78"""

# Read only specific columns / 특정 열만 읽기
df_partial = pd.read_csv(StringIO(csv_text), usecols=["name"])
print("usecols=['name']:")
print(df_partial)
print()

# Force a dtype on a column / 특정 열의 타입을 강제 지정
df_typed = pd.read_csv(StringIO(csv_text), dtype={"score": int})
print("dtype={'score': int}:")
print(df_typed.dtypes)
print()

# Save to CSV text -- using a buffer (no real file) to inspect the exact saved content
# CSV 텍스트로 저장 -- 버퍼를 이용해(실제 파일 아님) 저장될 내용을 정확히 확인
buffer = StringIO()
df_partial.to_csv(buffer, index=False)
print("to_csv() output:")
print(buffer.getvalue())

usecols=['name']:
      name
0    Alice
1      Bob
2  Charlie

dtype={'score': int}:
name       str
score    int64
dtype: object

to_csv() output:
name
Alice
Bob
Charlie



---
# 🧪 Small Examples

## Example 1 — Reading CSV: Custom Missing Values with na_values
*(Covers source section 2-1: na_values)*

**English:** Not every missing value looks like an empty cell. When a placeholder like `"-"` sits in a numeric column, pandas has no way to know it means "missing" unless you tell it with `na_values` — otherwise the whole column silently becomes text.  
**한글:** 모든 결측치가 빈 셀처럼 생긴 건 아닙니다. `"-"` 같은 표시가 숫자 열에 있으면, `na_values`로 알려주지 않는 한 pandas는 그것이 "결측"이라는 걸 알 방법이 없습니다 — 그러지 않으면 열 전체가 조용히 텍스트로 바뀝니다.

In [4]:
import pandas as pd
from io import StringIO

# --- Reading a REAL file (illustrative only -- no file exists here, so this stays commented out) ---
# --- 실제 파일 읽기 (설명용 -- 여기엔 파일이 없으므로 주석 처리) ---
# df = pd.read_csv("orders.csv")
# df = pd.read_csv("orders.csv", encoding="utf-8-sig")  # CSV saved by Excel (has a BOM) / Excel에서 저장한 CSV (BOM 포함)
# df = pd.read_csv("orders.csv", encoding="cp949")      # CSV saved on Korean Windows / Windows 한글 환경에서 저장한 CSV

# --- From here: a "virtual file" via StringIO so this cell actually runs end-to-end ---
# --- 여기부터: StringIO로 만든 "가상 파일" -- 이 셀은 실제로 끝까지 실행됨 ---
csv_text = """order_id,customer,amount,region
1001,Minsu,45000,Seoul
1002,Younghee,32000,Busan
1003,Junho,-,Seoul
1004,Seoyeon,78000,Incheon"""

# Without telling pandas that "-" means missing, the whole "amount" column becomes text
# "-"가 결측치라고 알려주지 않으면 "amount" 열 전체가 텍스트가 됨
df_raw = pd.read_csv(StringIO(csv_text))
print("Without na_values:")
print(df_raw.dtypes)
print()

# na_values tells pandas which strings should become NaN
# na_values는 어떤 문자열을 NaN으로 처리할지 pandas에게 알려줌
df = pd.read_csv(StringIO(csv_text), na_values=["-"])
print("With na_values=['-']:")
print(df)
print()
print(df.dtypes)

Without na_values:
order_id    int64
customer      str
amount        str
region        str
dtype: object

With na_values=['-']:
   order_id  customer   amount   region
0      1001     Minsu  45000.0    Seoul
1      1002  Younghee  32000.0    Busan
2      1003     Junho      NaN    Seoul
3      1004   Seoyeon  78000.0  Incheon

order_id      int64
customer        str
amount      float64
region          str
dtype: object


## Example 2 — Reading CSV: Dates, Thousands Separators, dtype & usecols
*(Covers source section 2-1: parse_dates, thousands, dtype, usecols, and the options summary table)*

**English:** Four more options handle almost everything else a messy CSV throws at you: `parse_dates` turns a text date into a real datetime, `thousands` strips separator commas before converting to a number, `dtype` forces a column to keep a leading zero (like a zip code), and `usecols` skips columns you don't need.  
**한글:** 네 가지 옵션이 지저분한 CSV가 던지는 나머지 대부분의 문제를 처리합니다: `parse_dates`는 텍스트 날짜를 진짜 datetime으로 바꾸고, `thousands`는 숫자로 변환하기 전에 구분 쉼표를 제거하며, `dtype`은 우편번호처럼 앞자리 0을 유지해야 하는 열의 타입을 강제하고, `usecols`는 필요 없는 열을 건너뜁니다.

### Frequently Used `read_csv()` Options / 자주 쓰는 옵션 요약

| Option / 옵션 | Role / 역할 | Example / 예시 |
|---|---|---|
| `sep` | change the delimiter / 구분자 변경 | `sep=";"` |
| `header` | header row position (default 0) / 헤더 행 위치 (기본 0) | `header=0` |
| `index_col` | use a column as the index / 특정 열을 인덱스로 | `index_col="id"` |
| `skiprows` | skip specific rows / 특정 행 건너뛰기 | `skiprows=[1,2]` |
| `nrows` | read only the first N rows / 앞에서 N행만 읽기 | `nrows=1000` |
| `na_values` | strings to treat as missing / 결측치로 처리할 값 | `na_values=["-","N/A"]` |
| `parse_dates` | auto-convert to datetime / 날짜로 자동 변환 | `parse_dates=["date"]` |
| `dtype` | force a column's type / 타입 강제 지정 | `dtype={"zip":str}` |
| `thousands` | strip thousands separators / 천단위 구분자 제거 | `thousands=","` |
| `usecols` | read only these columns / 필요한 열만 선택 | `usecols=["a","b"]` |

In [5]:
import pandas as pd
from io import StringIO

# parse_dates + thousands / 날짜 + 천단위 구분자
csv_text_1 = '''date,revenue
2024-01-15,"1,250,000"
2024-01-16,"980,000"
2024-01-17,"1,430,000"'''

df1 = pd.read_csv(StringIO(csv_text_1), parse_dates=["date"], thousands=",")
print("parse_dates + thousands:")
print(df1)
print(df1.dtypes)
print()

# dtype + usecols / 타입 강제 지정 + 필요한 열만 읽기
csv_text_2 = """id,name,zipcode,amount
1,Alice,06236,1000
2,Bob,03187,2000
3,Charlie,07321,1500"""

# Without dtype={"zipcode": str}, "06236" becomes the number 6236 -- the leading zero disappears
# dtype={"zipcode": str} 없이 읽으면 "06236"이 숫자 6236이 되어 앞의 0이 사라짐
df2 = pd.read_csv(StringIO(csv_text_2), dtype={"zipcode": str}, usecols=["name", "zipcode", "amount"])
print("dtype + usecols:")
print(df2)
print(df2.dtypes)

parse_dates + thousands:
        date  revenue
0 2024-01-15  1250000
1 2024-01-16   980000
2 2024-01-17  1430000
date       datetime64[us]
revenue             int64
dtype: object

dtype + usecols:
      name zipcode  amount
0    Alice   06236    1000
1      Bob   03187    2000
2  Charlie   07321    1500
name         str
zipcode      str
amount     int64
dtype: object


## Example 3 — Writing CSV: to_csv() and Verifying the Saved Content
*(Covers source section 2-1: to_csv)*

**English:** `to_csv()` is the mirror image of `read_csv()`. The one setting worth memorizing is `index=False` — without it, pandas saves the meaningless row-number index as an extra column in the file.  
**한글:** `to_csv()`는 `read_csv()`의 반대짝입니다. 꼭 외워둘 설정 하나는 `index=False`입니다 — 이게 없으면 pandas는 의미 없는 행 번호 인덱스까지 파일에 추가 열로 저장합니다.

In [6]:
import pandas as pd
from io import StringIO

df = pd.DataFrame({
    "name": ["Minsu", "Younghee"],
    "amount": [45000, 32000]
})

# The base pattern for saving to a REAL file (commented out -- no file exists here)
# 실제 파일로 저장하는 기본 패턴 (여기엔 파일이 없으므로 주석 처리)
# df.to_csv("output.csv", index=False, encoding="utf-8-sig")

# Inspect exactly what gets written, using a StringIO buffer instead of a real file
# StringIO 버퍼로 실제로 저장될 내용을 정확히 확인
buffer = StringIO()
df.to_csv(buffer, index=False)
print("with index=False:")
print(buffer.getvalue())

# Without index=False, a meaningless row-number column gets saved too
# index=False가 없으면 의미 없는 행 번호 열도 함께 저장됨
buffer2 = StringIO()
df.to_csv(buffer2)
print("without index=False:")
print(buffer2.getvalue())

with index=False:
name,amount
Minsu,45000
Younghee,32000

without index=False:
,name,amount
0,Minsu,45000
1,Younghee,32000



## Example 4 — Reading & Writing Excel: One Sheet, All Sheets, Multiple Sheets
*(Covers source section 2-2: read_excel / to_excel)*

**English:** Think of an Excel workbook as several DataFrames living in one file, one per tab. `sheet_name="Sales"` grabs one tab, `sheet_name=None` grabs every tab at once as a dict, and `pd.ExcelWriter` lets you write several DataFrames back out as separate tabs in a single file.  
**한글:** 엑셀 워크북은 하나의 파일 안에 탭마다 하나씩 들어있는 여러 개의 DataFrame이라고 생각하면 됩니다. `sheet_name="Sales"`는 탭 하나를 가져오고, `sheet_name=None`은 모든 탭을 한 번에 딕셔너리로 가져오며, `pd.ExcelWriter`는 여러 DataFrame을 하나의 파일에 각각 다른 탭으로 다시 저장하게 해줍니다.

In [7]:
import pandas as pd
from io import BytesIO

# Two sheets, as if they came from a real "sales.xlsx" workbook
# 실제 "sales.xlsx" 워크북에서 온 것 같은 두 개의 시트
sales_sheet = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "qty": [5, 20, 12]
})
returns_sheet = pd.DataFrame({
    "product": ["Mouse"],
    "reason": ["Defective"]
})

# Write both sheets into an in-memory "virtual" Excel file (BytesIO acts like a file)
# NOTE: if this errors on your machine, install the Excel engine: pip install openpyxl
# 두 시트를 메모리 상의 "가상" 엑셀 파일에 기록 (BytesIO가 파일처럼 동작)
# 참고: 오류가 나면 엑셀 엔진을 설치하세요: pip install openpyxl
excel_buffer = BytesIO()
with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
    sales_sheet.to_excel(writer, sheet_name="Sales", index=False)
    returns_sheet.to_excel(writer, sheet_name="Returns", index=False)

# Read a single sheet back -- sheet_name="Sales" or sheet_name=0 both work
# 특정 시트 하나만 다시 읽기 -- sheet_name="Sales"와 sheet_name=0 둘 다 동일
excel_buffer.seek(0)
one_sheet = pd.read_excel(excel_buffer, sheet_name="Sales")
print("Single sheet ('Sales'):")
print(one_sheet)
print()

# Read every sheet at once -> a dict of {sheet_name: DataFrame}
# 모든 시트를 한 번에 읽기 -> {시트이름: DataFrame} 형태의 딕셔너리
excel_buffer.seek(0)
all_sheets = pd.read_excel(excel_buffer, sheet_name=None)
print("type:", type(all_sheets))
print("sheet names:", list(all_sheets.keys()))
print()
print(all_sheets["Returns"])

Single sheet ('Sales'):
    product  qty
0    Laptop    5
1     Mouse   20
2  Keyboard   12

type: <class 'dict'>
sheet names: ['Sales', 'Returns']

  product     reason
0   Mouse  Defective


## Example 5 — Common Combo Patterns: Clean-on-Load & Combining Sheets
*(Covers source section 2-3: frequently-used comprehensive patterns)*

**English:** Pattern A stacks every option into one `read_csv()` call, so the DataFrame is analysis-ready the moment it's loaded. Pattern B turns a dict of sheets (from `sheet_name=None`) into one long table — the standard way to combine "one sheet per month" workbooks.  
**한글:** 패턴 A는 모든 옵션을 하나의 `read_csv()` 호출에 쌓아서, 불러오는 즉시 분석 가능한 DataFrame을 만듭니다. 패턴 B는 (`sheet_name=None`으로 얻은) 시트 딕셔너리를 하나의 긴 테이블로 바꿉니다 — "월별 시트 하나씩"인 워크북을 합치는 표준 방법입니다.

In [8]:
import pandas as pd
from io import StringIO

# Pattern A: apply every option in one clean read / 패턴 A: 옵션을 한번에 적용해서 깨끗하게 읽기
csv_text = '''order_date,customer,amount,note
2024-03-01,Minsu,"1,200,000",
2024-03-02,Younghee,N/A,refund pending
2024-03-03,Junho,"850,000",'''

df = pd.read_csv(
    StringIO(csv_text),
    parse_dates=["order_date"],
    thousands=",",
    na_values=["N/A"]
)
print("Pattern A -- one clean read:")
print(df)
print()

# Pattern B: combine multiple sheets/periods into one long table
# 패턴 B: 여러 시트(또는 여러 기간 데이터)를 하나로 합치기
sheets = {
    "Jan": pd.DataFrame({"product": ["A", "B"], "revenue": [100, 200]}),
    "Feb": pd.DataFrame({"product": ["A", "B"], "revenue": [150, 180]}),
}

frames = []
for month, sheet_df in sheets.items():
    sheet_df = sheet_df.copy()
    sheet_df["month"] = month
    frames.append(sheet_df)

combined = pd.concat(frames, ignore_index=True)
print("Pattern B -- sheets combined into one long table:")
print(combined)

Pattern A -- one clean read:
  order_date  customer     amount            note
0 2024-03-01     Minsu  1200000.0             NaN
1 2024-03-02  Younghee        NaN  refund pending
2 2024-03-03     Junho   850000.0             NaN

Pattern B -- sheets combined into one long table:
  product  revenue month
0       A      100   Jan
1       B      200   Jan
2       A      150   Feb
3       B      180   Feb


## Example 6 (Practice) — Fill in the Blanks
*(Practice built from the section 2-3 "clean-on-load" pattern -- the source PDF has no separate numbered practice problem for this section)*

**English:** Fill in each `________` blank below to load this messy CSV cleanly in one call. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a `NameError` instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워서 이 지저분한 CSV를 한 번의 호출로 깔끔하게 불러오세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 `NameError`가 나는데, 이는 의도된 동작입니다).

In [10]:
import pandas as pd
from io import StringIO

csv_text = '''signup_date,name,revenue,status
2024-05-01,Alice,"2,500,000",active
2024-05-02,Bob,N/A,pending
2024-05-03,Charlie,"1,800,000",active'''

# 1. Convert "signup_date" to a real date -- needs a LIST of column names
#    "signup_date"를 진짜 날짜로 변환 -- 열 이름의 리스트가 필요함
# 2. Strip thousands-separator commas from numbers -- needs the separator character as a string
#    숫자의 천단위 구분 쉼표 제거 -- 구분자 문자를 문자열로 전달
# 3. Treat "N/A" as a missing value -- needs a LIST of marker strings
#    "N/A"를 결측치로 처리 -- 마커 문자열의 리스트가 필요함
df = pd.read_csv(
    StringIO(csv_text),
    parse_dates=["signup_date"],
    thousands=",",
    na_values=["N/A"]
)
print(df)
print(df.dtypes)

  signup_date     name    revenue   status
0  2024-05-01    Alice  2500000.0   active
1  2024-05-02      Bob        NaN  pending
2  2024-05-03  Charlie  1800000.0   active
signup_date    datetime64[us]
name                      str
revenue               float64
status                    str
dtype: object


### 💡 Hint / 힌트
`parse_dates` needs a **list** of column names, e.g. `["signup_date"]`. `thousands` needs a single-character **string**, e.g. `","`. `na_values` needs a **list** of strings, e.g. `["N/A"]`.  
`parse_dates`에는 열 이름의 **리스트**가 필요합니다 (예: `["signup_date"]`). `thousands`에는 한 글자짜리 **문자열**이 필요합니다 (예: `","`). `na_values`에는 문자열의 **리스트**가 필요합니다 (예: `["N/A"]`).

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd
from io import StringIO

csv_text = '''signup_date,name,revenue,status
2024-05-01,Alice,"2,500,000",active
2024-05-02,Bob,N/A,pending
2024-05-03,Charlie,"1,800,000",active'''

df = pd.read_csv(
    StringIO(csv_text),
    parse_dates=["signup_date"],
    thousands=",",
    na_values=["N/A"]
)
print(df)
print(df.dtypes)

# Bob's revenue is NaN (float) because "N/A" was recognized as missing --
# that also forces the whole "revenue" column to float64, same rule as Section 1 Example 5.
# Bob의 revenue가 NaN(float)인 이유는 "N/A"가 결측치로 인식되었기 때문 --
# 이 때문에 "revenue" 열 전체가 float64가 됨. Section 1 Example 5와 같은 규칙.

---
# ⚠️ Common Mistakes

### Mistake 1 — Forgetting to declare a non-standard missing-value marker
**English:** A CSV cell containing `"-"` or `"N/A"` looks empty to a human, but pandas has no idea it means "missing" by default (only truly blank cells and a small built-in list are recognized automatically). If you skip `na_values`, the entire numeric column silently becomes text.  
**한글:** CSV 셀에 있는 `"-"`나 `"N/A"`는 사람 눈에는 비어 보이지만, pandas는 기본적으로 그것이 "결측"이라는 걸 모릅니다(진짜 빈 셀과 내장된 소수의 목록만 자동으로 인식됩니다). `na_values`를 빠뜨리면 숫자 열 전체가 조용히 텍스트가 됩니다.

**✅ Fix / 해결법:**  
Look at the raw file (or its `.dtypes` after a first read) for any non-numeric placeholder in a numeric column, and always pass it to `na_values=[...]`.  
원본 파일(또는 처음 읽은 뒤의 `.dtypes`)에서 숫자 열에 있는 비숫자 표시를 확인하고, 항상 `na_values=[...]`로 전달하세요.

### Mistake 2 — Forgetting `index=False` on `to_csv()`
**English:** `df.to_csv("file.csv")` without `index=False` saves the DataFrame's row-number index as an unnamed extra column. The next time someone (including future you) reads that file back in, they get a mysterious `"Unnamed: 0"` column full of numbers.  
**한글:** `index=False` 없이 `df.to_csv("file.csv")`를 호출하면 DataFrame의 행 번호 인덱스가 이름 없는 추가 열로 저장됩니다. 다음에 누군가(미래의 나를 포함해서) 이 파일을 다시 읽으면, 숫자로 가득한 정체불명의 `"Unnamed: 0"` 열을 만나게 됩니다.

**✅ Fix / 해결법:**  
Default to `index=False` every time you call `to_csv()`, unless you deliberately `set_index()`'d something meaningful first and actually want to keep it.  
`to_csv()`를 호출할 때는 의도적으로 의미 있는 것을 `set_index()`해서 정말로 유지하고 싶은 경우가 아니라면, 항상 기본적으로 `index=False`를 사용하세요.

### Mistake 3 — Letting an ID-like column get inferred as a number
**English:** Zip codes, phone numbers, and employee IDs that start with `0` (like `"06236"`) look numeric, so pandas infers them as integers by default — and integers can't have leading zeros, so `"06236"` silently becomes `6236`.  
**한글:** `"06236"`처럼 `0`으로 시작하는 우편번호, 전화번호, 사번은 숫자처럼 보이기 때문에 pandas는 기본적으로 이를 정수로 추론합니다 — 정수는 앞자리 0을 가질 수 없으므로 `"06236"`이 조용히 `6236`이 됩니다.

**✅ Fix / 해결법:**  
Force these columns to stay text with `dtype={"column_name": str}` in the `read_csv()` call itself — fixing a lost leading zero after the fact is much harder than preventing it.  
`read_csv()` 호출 자체에서 `dtype={"열이름": str}`로 이 열들을 텍스트로 강제 유지하세요 — 이미 사라진 앞자리 0을 나중에 되살리는 것보다 미리 막는 게 훨씬 쉽습니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Bundle `parse_dates` + `thousands` + `na_values` into a single `read_csv()` call instead of cleaning columns one by one afterward — this is the standard "clean-on-load" pattern used throughout real analyses.  
`parse_dates` + `thousands` + `na_values`를 나중에 하나씩 정리하지 말고 하나의 `read_csv()` 호출에 한 번에 묶으세요 — 실제 분석에서 표준으로 쓰이는 "불러오면서 정리하기" 패턴입니다.  
- `sheet_name=None` returns a `dict`, so you access each sheet with `sheets["SheetName"]`, and `pd.concat()` is the standard way to stack that dict into one long table.   
`sheet_name=None`은 `dict`를 반환하므로 `sheets["시트이름"]`으로 각 시트에 접근하고, 그 딕셔너리를 하나의 긴 테이블로 합칠 때는 `pd.concat()`이 표준 방법입니다.  
- Set `dtype={"col": str}` on any ID-like column *before* it's even loaded — fixing a stripped leading zero after the fact is much harder than preventing it.   
ID처럼 생긴 열은 불러오기 *전에* 항상 `dtype={"열": str}`을 지정하세요 — 이미 사라진 앞자리 0을 나중에 되살리는 것보다 미리 막는 게 훨씬 쉽습니다.  
- If Korean text looks garbled after `read_csv()`, try `encoding="utf-8-sig"` first (files saved by Excel), then `encoding="cp949"` (files saved on Korean Windows).   
`read_csv()` 이후 한글이 깨져 보인다면 먼저 `encoding="utf-8-sig"`(Excel에서 저장한 파일)를 시도하고, 그다음 `encoding="cp949"`(Windows 한글 환경에서 저장한 파일)를 시도하세요.

---
# 🔗 Related Concepts

```
Series & DataFrame     (Section 1 -- you already know how to build one by hand)
    ↓
Data I/O                ← you are here / 지금 여기 (Section 2)
    ↓
EDA                     (Section 3 -- the very first thing you run on a freshly-loaded DataFrame)
    ↓
Selection & Filtering   (Section 4)
    ↓
Data Cleaning           (Section 5 -- dropna/fillna pick up right where na_values left off)
    ↓
... GroupBy -> Merge -> Pivot -> Time Series -> BA Techniques
```

*How is today's topic connected to other concepts?*

**English:** Section 1 taught you how to build a DataFrame by hand from a dict or a list of dicts; `read_csv()` / `read_excel()` are just automated versions of that same construction, aimed at real files. The moment any of today's cells finishes running, the very next thing you'd naturally do is Section 3's `.head()` / `.info()` / `.describe()` to see what you just loaded. And `na_values` isn't the end of the missing-data story — it only decides what counts as missing *at load time*; Section 5 (`dropna()` / `fillna()`) picks up from there and decides what to actually *do* about it.

**한글:** 1번 섹션에서는 dict나 list of dicts로 DataFrame을 손으로 만드는 법을 배웠습니다. `read_csv()` / `read_excel()`은 그 동일한 구성 과정을 실제 파일을 대상으로 자동화한 것일 뿐입니다. 오늘 배운 셀 중 하나가 실행을 마치는 순간, 자연스럽게 다음으로 할 일은 3번 섹션의 `.head()` / `.info()` / `.describe()`로 방금 불러온 것을 살펴보는 것입니다. 그리고 `na_values`가 결측치 이야기의 끝은 아닙니다 — 이는 *불러오는 시점*에 무엇을 결측으로 볼지만 결정할 뿐이고, 5번 섹션(`dropna()` / `fillna()`)이 그 이후를 이어받아 결측치를 실제로 어떻게 *처리*할지 결정합니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** Your coffee shop's POS system exports one CSV per month. This month's export has a stray `"-"` for a day the register was closed, and every revenue number includes thousands-separator commas. Load it cleanly in a single `read_csv()` call, then prepare a cleaned copy to hand off.

**한글:** 커피숍의 POS 시스템은 매달 CSV 하나를 내보냅니다. 이번 달 내보내기 파일에는 계산대가 문을 닫은 날에 대해 `"-"`가 하나 섞여 있고, 모든 매출 숫자에는 천단위 구분 쉼표가 붙어 있습니다. 한 번의 `read_csv()` 호출로 깔끔하게 불러온 뒤, 정리된 사본을 전달할 준비를 하세요.

**To Do / 할 일**
- [x] Load the raw CSV text, converting dates and stripping thousands separators in the same call  
원본 CSV 텍스트를 불러오면서 동시에 날짜 변환과 천단위 구분자 제거하기  
- [x] Tell pandas that `"-"` means "closed" (missing), not literal text  
`"-"`가 "휴업"(결측)을 의미한다고 pandas에 알려주기 — 문자 그대로의 텍스트가 아니라  
- [x] Confirm the cleaned dtypes  
정리된 dtype 확인하기  
- [x] Save the cleaned DataFrame back out and inspect the saved text  
정리된 DataFrame을 다시 저장하고 저장될 내용 확인하기

In [11]:
import pandas as pd
from io import StringIO

pos_csv = '''date,revenue,transactions
2024-06-01,"1,250,000",142
2024-06-02,"980,000",98
2024-06-03,-,0
2024-06-04,"1,430,000",156'''

# Load + clean in one call / 한 번의 호출로 불러오면서 정리
daily = pd.read_csv(
    StringIO(pos_csv),
    parse_dates=["date"],
    thousands=",",
    na_values=["-"]
)
print(daily)
print()
print(daily.dtypes)
print()

# Save the cleaned version back out and inspect exactly what would be written
# 정리된 버전을 다시 저장하고 실제로 저장될 내용을 확인
buffer = StringIO()
daily.to_csv(buffer, index=False)
print("Cleaned CSV ready to save:")
print(buffer.getvalue())

# Quick sanity check: how many days was the shop actually open?
# 간단한 검증: 실제로 영업한 날은 며칠인가?
print("Days open:", daily["revenue"].notna().sum(), "/ Days closed:", daily["revenue"].isna().sum())

        date    revenue  transactions
0 2024-06-01  1250000.0           142
1 2024-06-02   980000.0            98
2 2024-06-03        NaN             0
3 2024-06-04  1430000.0           156

date            datetime64[us]
revenue                float64
transactions             int64
dtype: object

Cleaned CSV ready to save:
date,revenue,transactions
2024-06-01,1250000.0,142
2024-06-02,980000.0,98
2024-06-03,,0
2024-06-04,1430000.0,156

Days open: 3 / Days closed: 1


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
Reading a file is rarely as simple as `pd.read_csv(path)` once real-world messiness shows up: `na_values` tells pandas which text markers actually mean "missing," `parse_dates` and `thousands` convert dates and comma-separated numbers on the way in, and `dtype` protects ID-like columns from losing their leading zeros. Excel works the same way but one level up — a workbook is several DataFrames in one file, accessed by sheet name (`sheet_name="X"`) or all at once as a dict (`sheet_name=None`), and `pd.ExcelWriter` writes several sheets back out in one file. `to_csv(index=False)` is the standard way to save a clean CSV without an extra row-number column. The single most valuable habit from this notebook is the "clean-on-load" pattern: stack every relevant option into one `read_csv()` call instead of fixing columns one at a time after the fact.

**한글**
실제 파일의 지저분함이 드러나기 시작하면 파일을 읽는 일은 `pd.read_csv(path)`처럼 간단하지 않습니다: `na_values`는 어떤 텍스트 표시가 실제로 "결측"을 의미하는지 pandas에게 알려주고, `parse_dates`와 `thousands`는 불러오는 과정에서 날짜와 쉼표 구분 숫자를 변환하며, `dtype`은 ID처럼 생긴 열이 앞자리 0을 잃지 않도록 보호합니다. Excel도 원리는 같지만 한 단계 위입니다 — 워크북은 하나의 파일 안에 여러 개의 DataFrame이 들어있는 것이고, 시트 이름으로(`sheet_name="X"`) 하나씩 또는 한 번에 딕셔너리로(`sheet_name=None`) 접근하며, `pd.ExcelWriter`로 여러 시트를 한 파일에 다시 저장할 수 있습니다. `to_csv(index=False)`는 불필요한 행 번호 열 없이 깨끗한 CSV를 저장하는 표준 방법입니다. 이 노트북에서 가장 가치 있는 습관은 "불러오면서 정리하기" 패턴입니다 — 열을 나중에 하나씩 고치는 대신, 관련 옵션을 전부 하나의 `read_csv()` 호출에 쌓아 올리세요.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Reading and writing files is where messiness meets structure — a handful of `read_csv()` / `read_excel()` parameters, applied at load time, turn inconsistent real-world exports into analysis-ready DataFrames in a single line.

> 파일을 읽고 쓰는 것은 지저분함이 구조를 만나는 지점입니다 — `read_csv()` / `read_excel()`의 몇 가지 파라미터를 불러오는 시점에 적용하면, 일관성 없는 실제 내보내기 파일이 한 줄만에 분석 가능한 DataFrame이 됩니다.

---
# ❓ Review Questions

**Q1.** What does `na_values=["-"]` actually do, and what happens to a numeric column if you forget it while the column contains a stray `"-"`?
**Q1.** `na_values=["-"]`는 실제로 무엇을 하며, 이를 빠뜨렸는데 숫자 열에 `"-"`가 하나 섞여 있으면 어떻게 될까요?

na_values=["-"] tells Pandas to treat "-" as a missing value (NaN). Without it, a numeric column containing "-" may be read as object instead of a numeric dtype.  
na_values=["-"]는 "-"를 결측값인 NaN으로 처리하라는 뜻입니다. 이걸 빼면 숫자 열에 "-"가 섞여 있어서 해당 열이 숫자가 아니라 object로 읽힐 수 있습니다.

**Q2.** Which single `read_csv()` parameter prevents a zip code like `"06236"` from losing its leading zero?
**Q2.** `read_csv()`의 어떤 파라미터 하나가 `"06236"` 같은 우편번호가 앞자리 0을 잃지 않도록 막아주나요?

Use the dtype parameter.
read_csv()에서 처음부터 문자열로 읽으려면 dtype을 사용합니다.

**Q3.** What type of object does `pd.read_excel("file.xlsx", sheet_name=None)` return, and how do you pull one specific sheet out of it?
**Q3.** `pd.read_excel("file.xlsx", sheet_name=None)`은 어떤 타입의 객체를 반환하며, 그중 특정 시트 하나는 어떻게 꺼내나요?

sheet_name=None returns a dictionary of DataFrames. You can access a specific sheet using its sheet name as the key.  
sheet_name=None을 사용하면 딕셔너리 형태로 모든 시트를 반환합니다.

**Q4.** Why should you almost always pass `index=False` to `to_csv()`?
**Q4.** `to_csv()`를 호출할 때 왜 거의 항상 `index=False`를 넘겨야 하나요?

index=False prevents Pandas from writing the DataFrame's index as an extra column in the CSV.  
index=False는 DataFrame의 인덱스를 CSV에 불필요한 컬럼으로 저장하지 않도록 합니다.

**Q5.** Name the three `read_csv()` parameters used in the "clean-on-load" pattern from Example 5, and what each one does.
**Q5.** Example 5의 "불러오면서 정리하기" 패턴에서 사용된 `read_csv()` 파라미터 3가지와 각각의 역할을 말해보세요.

parse_dates=["order_date"] converts the column to a date type.  
thousands="," handles commas in numbers such as "1,234".  
na_values=["N/A"] converts "N/A" into NaN.

parse_dates=["order_date"]는 해당 열을 날짜 타입으로 변환합니다.  
thousands=","는 "1,234" 같은 숫자의 천 단위 구분자를 처리합니다.  
na_values=["N/A"]는 "N/A"를 NaN으로 처리합니다.

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*